In [1]:
#1 The Data

In [2]:
import torch
from torch_geometric.datasets import MovieLens100K

In [3]:
dataset = MovieLens100K(root = "./data")

In [4]:
# Data contains 1 graph
len(dataset)

1

In [5]:
# Graph info
# Note that we have a split of 80 000 training positives
# And 20 000 held-out positive test edges
data = dataset[0]
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)


In [6]:
print(data.node_types)

['movie', 'user']


In [7]:
print(data.edge_types)

[('user', 'rates', 'movie'), ('movie', 'rated_by', 'user')]


In [8]:
# Movie vectors represent movie genres (one movie can belong to several genres).
movie_x = data["movie"].x
print(movie_x[:5])

tensor([[0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])


In [9]:
# Set all seeds to further create reproducible split, negative edges and model:

import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [10]:
#2 GCN baseline

In [11]:
#2.1. We need to transform the graph, because GCN works with nodes of one type.
#We first transform edge index: shift movie indices by the number of the user nodes,
#and then unite them with user indices in one row.

In [15]:
# The current edge index from the training set (80000 edges: 1st row - users, 2nd row - movies):
edges = data["user", "rates", "movie"].edge_index
print(edges)
print(edges.shape)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])
torch.Size([2, 80000])


In [13]:
# Take movie indices and shift them by the number of the user nodes
movies = edges[1]
number_users = data["user"].num_nodes
shifted_movies = movies + number_users
users = edges[0]
edges = torch.stack([users, shifted_movies], dim=0)
print(edges)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [ 943,  944,  945,  ..., 2130, 2170, 2272]])


In [14]:
# The current pos_edge_index contain a one-directional relation genre -> movie
# Let's reverse the relation and add movie -> genre:
reverse_edges = edges.flip(0)
message_passing_edges = torch.cat([edges, reverse_edges], dim=1)
print(message_passing_edges.shape)

torch.Size([2, 160000])


In [23]:
#2.2. Sample negative examples
# We have positive edges (the edges that really exist).
# We now need to sample negative edges (fake edges).
# Then we unite these edges in a new training set,
# so that the recommender predict as mush as possible of the positive (really existing) edges.

In [18]:
number_movies = data["movie"].num_nodes
number_pos = edges.shape[1]
print(number_pos)

80000


In [16]:
# We also need to make sure that none of the 20,000 held-out positive test edges will be sampled as a training negative.
# In other words, we exclude all known interactions when generating negatives.
# Create all known interactions:
all_pos_edges = torch.cat([edges, data["user", "rates", "movie"].edge_label_index], dim=1)
print(all_pos_edges.shape)

torch.Size([2, 100000])


In [20]:
# Sample negative edges excluding all positive edges
from torch_geometric.utils import negative_sampling
neg_edges = negative_sampling(all_pos_edges, (number_users, number_movies), number_pos)
print(neg_edges)
print(neg_edges.shape)

tensor([[ 188,  397,  248,  ...,    0,  127,  100],
        [ 983, 1549,  690,  ...,  521,  269,  129]])
torch.Size([2, 80000])


In [22]:
# Then take movie indices for the negative edge index
# and shift them by the number of users (like we did for positive edges)
neg_movies = neg_edges[1]
shifted_neg_movies = neg_movies + number_users
neg_users = neg_edges[0]
neg_edges = torch.stack([neg_users, neg_movies], dim=0)
print(neg_edges)

tensor([[ 188,  397,  248,  ...,    0,  127,  100],
        [ 983, 1549,  690,  ...,  521,  269,  129]])


In [1046]:
# Now we create supervision set (concatenate the pos and neg edges)
# and also create labels 

In [53]:
supervis_edges = torch.cat([edges, neg_edges], dim=1)
print(supervis_edges)
print(supervis_edges.shape)

tensor([[  0,   0,   0,  ...,   0, 127, 100],
        [  0,   1,   2,  ..., 521, 269, 129]])
torch.Size([2, 160000])


In [54]:
y_1 = torch.ones(edges.shape[1])
y_2 = torch.zeros(neg_edges.shape[1])
y_train = torch.cat([y_1, y_2])
print(y_train.shape)

torch.Size([160000])


In [27]:
#2.4. Create GCN Recommender
# The GCN Recommender first transform the user and movie nodes in a way that they have the same number of features:
# 24 user features -> 64
# 18 user features -> 18
# This is done by linear layers of the GCN recommender.
# Then two GCN layers are stacked

# After the second GCN layer we concatenate the user embedding with the movie embedding, and - via a linear layer -
# get a single prediction whether an edge exists or if it is a fake.

# Note that there are message passing edges and training edges:
# message passing edges are used to build the representation of the nodes
# message training are used to see if the obtained representations perform well to predict true vs false edges

In [26]:
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GCNConv

In [55]:
class GCNModel(nn.Module):
    def __init__(self,
                 init_dim_users,
                 init_dim_movies,
                 dim_unified,
                 hidden_dim,
                 embed_dim,
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.transform_users = nn.Linear(init_dim_users, dim_unified)
        self.transform_movies = nn.Linear(init_dim_movies, dim_unified)
        self.gcn1 = GCNConv(dim_unified, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, embed_dim)
        self.head = nn.Linear(embed_dim * 2, prediction_binary)
    def forward(self, users, movies, message_pass_adj, supervis_adj):
        #Your code goes here#
        u = self.transform_users(users)
        m = self.transform_movies(movies)
        x = torch.cat([u, m], dim=0)
        x = self.gcn1.forward(x, message_pass_adj)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.gcn2.forward(x, message_pass_adj)
        x = F.dropout(x, p=0.5, training=self.training)
        x = torch.cat([ x[supervis_adj[0]], x[supervis_adj[1]] ], dim=1)
        x = self.head(x)
        return x

In [56]:
feature_users = data["user"].x.shape[1] #24
feature_movies = data["movie"].x.shape[1] #18

In [57]:
model = GCNModel(init_dim_users=feature_users, init_dim_movies=feature_movies, dim_unified=32, hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [35]:
#2.5. Train GCN Recommender
# Demonstrate binary cross-entropy loss
# as well as the accuracy

In [58]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model.forward(users=data["user"].x, movies=data["movie"].x,
                         message_pass_adj=message_passing_edges, supervis_adj=supervis_edges)
    l = loss(pred.squeeze(1), y_train)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        pred = (pred.sigmoid() >= 0.5).float()
        accuracy = (pred.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.6999853253364563, accuracy: 0.49645623564720154
epoch 20, CEL: 0.5580364465713501, accuracy: 0.7219499945640564
epoch 40, CEL: 0.5344812273979187, accuracy: 0.7385749816894531
epoch 60, CEL: 0.5337744951248169, accuracy: 0.7410749793052673
epoch 80, CEL: 0.5268298983573914, accuracy: 0.7439000010490417
epoch 100, CEL: 0.5209017395973206, accuracy: 0.7487062215805054
epoch 120, CEL: 0.5198889374732971, accuracy: 0.7505249977111816
epoch 140, CEL: 0.5226871371269226, accuracy: 0.7457812428474426
epoch 160, CEL: 0.5155782699584961, accuracy: 0.7509812712669373
epoch 180, CEL: 0.5141372084617615, accuracy: 0.7517374753952026
epoch 200, CEL: 0.5106068849563599, accuracy: 0.7537875175476074


In [1056]:
# 2.6 Evaluate GCN Recommender
# First we prepare an evaluation set - in the same way as for the training set

In [46]:
# Shift the indices
test_edges = data["user", "rates", "movie"].edge_label_index # The current edge index for the test data (20000 edges)
test_movies = test_edges[1]
shifted_test_movies = test_movies + number_users
test_users = (data["user", "rates", "movie"].edge_label_index)[0]
test_edges = torch.stack([test_users, shifted_test_movies], dim=0)
print(test_edges)

tensor([[   0,    0,    0,  ...,  458,  459,  461],
        [ 948,  952,  954,  ..., 1876,  952, 1624]])


In [47]:
# Sample negative edges

In [48]:
test_number_pos = test_edges.shape[1]
print(test_number_pos)

20000


In [49]:
test_neg_edges = negative_sampling(
    #note that as before we use all positive edges to sample negatives:
    all_pos_edges,
    (number_users, number_movies),
    test_number_pos)
print(test_neg_edges)
print(test_neg_edges.shape)

tensor([[ 218,  824,  627,  ...,  326,  342,   99],
        [ 702,  896,  899,  ..., 1042,  750, 1652]])
torch.Size([2, 20000])


In [50]:
shifted_test_neg_movies = test_neg_edges[1] + number_users
test_neg_users = test_neg_edges[0]
test_neg_edges = torch.stack([test_neg_users, shifted_test_neg_movies], dim=0)
print(test_neg_edges)

tensor([[ 218,  824,  627,  ...,  326,  342,   99],
        [1645, 1839, 1842,  ..., 1985, 1693, 2595]])


In [51]:
# Create test set

In [60]:
test_supervis_edges = torch.cat([test_edges, test_neg_edges], dim=1)
print(test_supervis_edges)
print(test_supervis_edges.shape)

tensor([[   0,    0,    0,  ...,  326,  342,   99],
        [ 948,  952,  954,  ..., 1985, 1693, 2595]])
torch.Size([2, 40000])


In [61]:
y_test_1 = torch.ones(test_edges.shape[1])
y_test_2 = torch.zeros(test_neg_edges.shape[1])
y_test = torch.cat([y_test_1, y_test_2])
print(y_test.shape)

torch.Size([40000])


In [1065]:
#2.6.3. Launch evaluation

In [1066]:
model.eval()
with torch.no_grad():
    pred_test = model.forward(users=data["user"].x, movies=data["movie"].x,
                              # note that we use message passing edges from the training set
                              message_pass_adj=message_passing_edges,
                              supervis_adj=test_supervis_edges)
    l_test = loss(pred_test.squeeze(1), y_test)
    pred_test = (pred_test.sigmoid() >= 0.5).float()
    accuracy_test = (predictions_test.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test}, test accuracy: {accuracy_test}")

test CEL: 0.5145467519760132, test accuracy: 0.7501000165939331


In [1067]:
#3. R-GCN

In [1068]:
# 3.1. Unlike GCN, RGCN can handle heterogeneous graphs -> can handle nodes of different sizes
# Hence we do not need transform the initial graph (to shift film indices) like we did for GCN
# Instead for training we use the initial edge index

In [1069]:
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)


In [1070]:
pos_edge_index_relational = data["user", "rates", "movie"].edge_index
print(pos_edge_index_relational)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])


In [1071]:
# We will also need relations (movie ratings from 1 to 5) and we will interpret them as different edge types:
# so RGCN is supposed to accumulate information from neughbours via different edge types: 1, 2, 3, 4, and 5.

In [1072]:
relations = data["user", "rates", "movie"].rating.long() - 1
print(relations)

tensor([4, 2, 3,  ..., 2, 2, 2])


In [1073]:
# 3.2. Create relational training data
# We already created the training data for GCN, but to do this we shifted the film indices
# For RGCN we do not need to transform the graph, so we to obtain the training data for RGCN we can just shift the indices back:

In [1074]:
print(training_edge_index) #this is the training data with shifted indices for GCN

tensor([[   0,    0,    0,  ...,  888,  595,  772],
        [ 943,  944,  945,  ..., 1406, 1005, 1848]])


In [1075]:
# We shift the indices back by the number of users and get the training data for GCN
transform_back = training_edge_index[1] - number_users
training_users = training_edge_index[0]
training_edge_index_relational = torch.stack([training_users, transform_back], dim=0)
print(training_edge_index_relational)

tensor([[  0,   0,   0,  ..., 888, 595, 772],
        [  0,   1,   2,  ..., 463,  62, 905]])


In [1076]:
# For the labels we just need the same 80 000 ones and 80 000 zeros as for the GCN 
y_train_relational = y_train
print(y_train_relational)
print(y_train.shape)

tensor([1., 1., 1.,  ..., 0., 0., 0.])
torch.Size([160000])


In [1077]:
#3.3. Create RGCN Recommender

In [1078]:
# RGCN can take a tuple of two nodes (source, target), which do not necessary coincide in the number of features
# So we can feed a tuple of users (24 features) and movies (18 features) directly to the RGCN layer

# The RGCN layer returns the embedding of the target node (in the recommender below - the movie)
# After we get the movie embedding we concatenate it with the user embedding (which was not changed while training)
# and get a prediction if the edge really exists or it is a fake.

# The recommender below utilizes two RGCN layers

In [1079]:
from torch_geometric.nn import RGCNConv

In [1080]:
class RGCNModel(nn.Module):
    def __init__(self, dim_users,
                 dim_movies,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.rgcn1 = RGCNConv((dim_users, dim_movies), hidden_dim, num_relations=5)
        self.rgcn2 = RGCNConv((dim_users, hidden_dim), embed_dim, num_relations=5)
        self.head = nn.Linear(embed_dim + feature_users, prediction_binary)
    def forward(self, user, movie, message_pass_adj_relational, relation_types, training_adj_relational):
        #Your code goes here#
        m_emb = self.rgcn1.forward((user, movie), message_pass_adj_relational, relation_types)
        m_emb = F.relu(m_emb)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        m_emb = self.rgcn2.forward((user, m_emb), message_pass_adj_relational, relation_types)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        x = torch.cat([ user[training_adj_relational[0]], m_emb[training_adj_relational[1]] ], dim=1)
        x = self.head(x)
        return x

In [1081]:
model_rgcn = RGCNModel(dim_users=feature_users, dim_movies=feature_movies, hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model_rgcn.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [1082]:
#3.4. Train RGCN Recommender

In [1083]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model_rgcn.forward(user=user, movie=movie, message_pass_adj_relational=pos_edge_index_relational,
                              relation_types=relations, training_adj_relational=training_edge_index_relational)
    l = loss(pred.squeeze(1), y_train_relational)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        predictions = (pred.sigmoid() >= 0.5).float()
        accuracy = (predictions.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.7074543237686157, accuracy: 0.4921875
epoch 20, CEL: 0.5434960722923279, accuracy: 0.7308874726295471
epoch 40, CEL: 0.5176921486854553, accuracy: 0.7454875111579895
epoch 60, CEL: 0.5136110186576843, accuracy: 0.7503625154495239
epoch 80, CEL: 0.5077170133590698, accuracy: 0.7538062334060669
epoch 100, CEL: 0.5055274963378906, accuracy: 0.7539937496185303
epoch 120, CEL: 0.5028854608535767, accuracy: 0.7568437457084656
epoch 140, CEL: 0.5026919841766357, accuracy: 0.7560562491416931
epoch 160, CEL: 0.501515805721283, accuracy: 0.756668746471405
epoch 180, CEL: 0.5021256804466248, accuracy: 0.7555687427520752
epoch 200, CEL: 0.5015716552734375, accuracy: 0.7572125196456909


In [1084]:
#3.5. Evaluate RGCN Recommender

In [1085]:
# We use initial edge index for the test data
# as well as relations (movie ratings).

In [1086]:
test_pos_edge_index_relational = data["user", "rates", "movie"].edge_label_index
print(test_pos_edge_index_relational)
print(test_pos_edge_index_relational.shape)

tensor([[  0,   0,   0,  ..., 458, 459, 461],
        [  5,   9,  11,  ..., 933,   9, 681]])
torch.Size([2, 20000])


In [1087]:
test_relations = data["user", "rates", "movie"].edge_label.long() - 1
print(test_relations)

tensor([4, 2, 4,  ..., 2, 2, 4])


In [1088]:
# To create test data we shift back the movie indices of the GCN test data

In [1089]:
print(test_edge_index)
print(test_edge_index.shape)

tensor([[   0,    0,    0,  ...,  753,  874,  357],
        [ 948,  952,  954,  ..., 1132, 2505, 1826]])
torch.Size([2, 40000])


In [1090]:
test_transform_back = test_edge_index[1] - number_users
test_users = test_edge_index[0]
test_edge_index_relational = torch.stack([test_users, test_transform_back], dim=0)
print(test_edge_index_relational)
print(test_edge_index_relational.shape)

tensor([[   0,    0,    0,  ...,  753,  874,  357],
        [   5,    9,   11,  ...,  189, 1562,  883]])
torch.Size([2, 40000])


In [1091]:
# We use 20 000 ones and 20 000 zeros as for the GCN before

In [1092]:
y_test_relational = y_test
print(y_test_relational)
print(y_test.shape)

tensor([1., 1., 1.,  ..., 0., 0., 0.])
torch.Size([40000])


In [1093]:
#3.6. Launch Evaluation

In [1094]:
model_rgcn.eval()
with torch.no_grad():
    pred_test_relational = model_rgcn.forward(user=user, movie=movie,
                                              #note that we use positive edge index from the training to prevent the leakage of the answers:
                                              message_pass_adj_relational=pos_edge_index_relational,
                                              relation_types=relations, 
                                              training_adj_relational=test_edge_index_relational)
    l_test_relational = loss(pred_test_relational.squeeze(1), y_test_relational)
    predictions_test_relational = (pred_test_relational.sigmoid() >= 0.5).float()
    accuracy_test_relational = (predictions_test_relational.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test_relational}, test accuracy: {accuracy_test_relational}")

test CEL: 0.5083792209625244, test accuracy: 0.7505999803543091


In [1095]:
#4 RCGN + genres
# We now upgrade the RGCN recommender, using 5 movie ratings as edge type, with an additional relation - genre relation
# There are 18 genres, and one movie can belong to more than 1 genre

In [1096]:
#4.1. Create genre edge index 

In [1097]:
# Since there is no edge relation in the initial data, we need to create genre edge index first.
# It is possible to do that by reconstructing information about genres from movie embeddings.
# It is possible to do because the features of the movie embeddings represent genres:
# the embedding has 18 features (0 and 1) such that 1 stands for "the movie has this genre":

print(movie_x[:5])

tensor([[0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])


In [1098]:
# With torch nonzero we define the exact place of each 1 in movie_x:
movie_ids, genre_ids = movie_x.nonzero(as_tuple=True)
# Now we can create a new edge type in the data
data["movie", "has_genre", "genre"].edge_index = torch.stack([genre_ids, movie_ids], dim=0)
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={ edge_index=[2, 2891] }
)


In [1099]:
genre_idx = data["movie", "has_genre", "genre"].edge_index
print(genre_idx)

tensor([[   2,    3,    4,  ...,   13,    4,    7],
        [   0,    0,    0,  ..., 1679, 1680, 1681]])


In [1100]:
# all genres are already in genre_idx, all genre-to-movie edges can use one semantic relation:
# genre --describes--> movie

genre_labels = torch.zeros(genre_idx.size(1), dtype=torch.long)
print(genre_labels)
print(genre_labels.shape)

tensor([0, 0, 0,  ..., 0, 0, 0])
torch.Size([2891])


In [1101]:
#4.2 Create genre nodes
# Note that we do not have genre nodes
# So we create them by initializing embedding direct in the Recommender
# We initialize 18 embeddings in the Recommender and they are learned during training

In [1102]:
num_genres = data["movie"].x.size(1)
print(num_genres)

18


In [1103]:
num_genres = data["movie"].x.size(1)  # 18
data["genre"].num_nodes = num_genres
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  genre={ num_nodes=18 },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={ edge_index=[2, 2891] }
)


In [1104]:
num_genres = torch.arange(num_genres) # We trainsform the integer 18 into the tensor,
# because the tensor-format is needed for initializing embeddings via nn.Embedding (see below)
print(num_genres)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])


In [1105]:
#4.3 Create RGCN-genre Recommender
# Genre embeddings are initiolized via nn.Embedding

# There are 4 RGCN layers: 1, 2, 3, 4
# RGCN layers 1 and 2 give us two types of movie embeddings: the first one is got from "user->movie" relation,
# the other form "genre->movie" relation
# We then sum up these two embeddings (they have the same size)
# Levels 3 and 4 repeat the operation

# Finally, as before, we concatenate the initial user embedding with the obtained movie embedding along with the training edges,
# and get a prediction, whether the edge exists or it is a fake.

In [1106]:
class RGCNModel_genre(nn.Module):
    def __init__(self, dim_users,
                 dim_movies,
                 dim_genres,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.genre = torch.nn.Embedding(num_embeddings=dim_movies, embedding_dim=dim_genres)
        self.rgcn1 = RGCNConv((dim_users, dim_movies), hidden_dim, num_relations=5)
        self.rgcn2 = RGCNConv((dim_genres, dim_movies), hidden_dim, num_relations=1)
        self.rgcn3 = RGCNConv((dim_users, hidden_dim), embed_dim, num_relations=5)
        self.rgcn4 = RGCNConv((dim_genres, hidden_dim), embed_dim, num_relations=1)
        self.head = nn.Linear(embed_dim + feature_users, prediction_binary)
    def forward(self,
                user, movie, genre,
                message_pass_adj_relational_user_movie, message_pass_adj_relational_genre_movie,
                relation_types_ratings, relation_types_genres,
                training_adj_relational):
        #Your code goes here#
        m_emb_1 = self.rgcn1.forward((user, movie), message_pass_adj_relational_user_movie, relation_types_ratings)
        genre = self.genre(genre)
        m_emb_2 = self.rgcn2.forward((genre, movie), message_pass_adj_relational_genre_movie, relation_types_genres)
        m_emb = m_emb_1 + m_emb_2
        m_emb = F.relu(m_emb)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        m_emb_1 = self.rgcn3.forward((user, m_emb), message_pass_adj_relational_user_movie, relation_types_ratings)
        m_emb_2 = self.rgcn4.forward((genre, m_emb), message_pass_adj_relational_genre_movie, relation_types_genres)
        m_emb = m_emb_1 + m_emb_2
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        x = torch.cat([ user[training_adj_relational[0]], m_emb[training_adj_relational[1]] ], dim=1)
        x = self.head(x)
        return x

In [1107]:
model_rgcn_genre = RGCNModel_genre(dim_users=feature_users, dim_movies=feature_movies, dim_genres=30,
                                   hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model_rgcn_genre.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [1108]:
#4.4. Train RGCN-genre Recommender

In [1109]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model_rgcn_genre.forward(user=user, movie=movie, genre=num_genres,
                                    message_pass_adj_relational_user_movie=pos_edge_index_relational,
                                    message_pass_adj_relational_genre_movie=genre_idx,
                                    relation_types_ratings=relations, relation_types_genres=genre_labels,
                                    training_adj_relational=training_edge_index_relational)
    l = loss(pred.squeeze(1), y_train_relational)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        predictions = (pred.sigmoid() >= 0.5).float()
        accuracy = (predictions.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.8569591641426086, accuracy: 0.48880624771118164
epoch 20, CEL: 0.5856226086616516, accuracy: 0.6938062310218811
epoch 40, CEL: 0.5495964288711548, accuracy: 0.7246875166893005
epoch 60, CEL: 0.534885048866272, accuracy: 0.7330187559127808
epoch 80, CEL: 0.5300530195236206, accuracy: 0.7394187450408936
epoch 100, CEL: 0.522168755531311, accuracy: 0.7438562512397766
epoch 120, CEL: 0.5179834961891174, accuracy: 0.7482812404632568
epoch 140, CEL: 0.5148047804832458, accuracy: 0.7476249933242798
epoch 160, CEL: 0.5116759538650513, accuracy: 0.7517499923706055
epoch 180, CEL: 0.5096861124038696, accuracy: 0.753000020980835
epoch 200, CEL: 0.5084901452064514, accuracy: 0.7526062726974487


In [1110]:
#4.5. Evaluate RGCN-genre recommender

In [1111]:
model_rgcn_genre.eval()
with torch.no_grad():
    pred_test_relational_genre = model_rgcn_genre.forward(user=user, movie=movie, genre=num_genres,
                                                          message_pass_adj_relational_user_movie=pos_edge_index_relational,
                                                          message_pass_adj_relational_genre_movie=genre_idx,
                                                          relation_types_ratings=relations, relation_types_genres=genre_labels,
                                                          training_adj_relational=test_edge_index_relational)
    l_test_relational_genre = loss(pred_test_relational_genre.squeeze(1), y_test_relational)
    predictions_test_relational_genre = (pred_test_relational_genre.sigmoid() >= 0.5).float()
    accuracy_test_relational_genre = (predictions_test_relational_genre.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test_relational_genre}, test accuracy: {accuracy_test_relational_genre}")

test CEL: 0.5130391716957092, test accuracy: 0.7488250136375427
